<a href="https://colab.research.google.com/github/MichaelangeloVelalopoulos/diploma-energy-market/blob/main/notebooks/XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ======================================================================================================================
# XGBoost MTU (96x15min) Baseline (Leakage-Safe) + D-1 BM Price Alignment + Metrics + CSV Export
#
# Goal:
#   Predict intraday MCP(t) at 15-min resolution (96 MTUs) using ONLY ex-ante features at time t.
#
# Key leakage-safe rules implemented:
#   1) Chronological split by DAYS (train -> val -> test)
#   2) D-1 BM PRICE (and optionally BMmDAMMCP) aligned by "same timestamp previous day" via merge
#   3) No use of MCP(t) or any intraday realized variable from the target day as feature
#
# Model:
#   Option A (recommended): Predict residual = MCP(t) - DAM_MCP(t)
#       MCP_hat(t) = DAM_MCP(t) + residual_hat(t)
#   This stabilizes learning and respects that DAM is a strong anchor.
#
# Outputs:
#   /content/xgb_mtu_preds.csv
#   /content/xgb_mtu_metrics.csv
# ======================================================================================================================

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

from dataclasses import dataclass
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# ---------------------------
# CONFIG
# ---------------------------
@dataclass
class CFG:
    DATA_PATH: str = "/content/Final2026.csv"
    TS_COL: str = "DELIVERY_MTU"
    TARGET_COL: str = "MCP"
    DAM_COL: str = "DAM_MCP"

    FREQ_MIN: int = 15
    MTU_PER_DAY: int = 96

    # Blocks for optional categorical structure (still ex-ante)
    BLOCK_HOURS: int = 4  # 6 blocks/day (0..5)
    BLOCKS_PER_DAY: int = 6

    # Splits (days)
    TRAIN_FRAC: float = 0.70
    VAL_FRAC: float   = 0.15
    TEST_FRAC: float  = 0.15

    # What the XGB learns:
    #  - "residual": y = MCP - DAM (recommended)
    #  - "mcp":      y = MCP directly
    TARGET_MODE: str = "residual"

    # Optional: exclude BM imbalance price if you suspect it is ex-post.
    INCLUDE_BM_IMBALANCE_PRICE: bool = True

    # XGB params
    N_EST: int = 2500
    LR: float = 0.03
    MAX_DEPTH: int = 6
    SUBSAMPLE: float = 0.9
    COLSAMPLE: float = 0.9
    REG_LAMBDA: float = 1.0
    SEED: int = 42

    # Export
    OUT_PRED_CSV: str = "/content/xgb_mtu_preds.csv"
    OUT_METRICS_CSV: str = "/content/xgb_mtu_metrics.csv"

cfg = CFG()
np.random.seed(cfg.SEED)

# ---------------------------
# METRICS
# ---------------------------
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_r2(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    if np.std(y_true) < 1e-12:
        return np.nan
    return float(r2_score(y_true, y_pred))

def smape(y_true, y_pred, eps=1e-6) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)), eps)
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)

def safe_mape(y_true, y_pred, thresh=10.0, eps=1e-6) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.abs(y_true) >= thresh
    if mask.sum() == 0:
        return np.nan
    denom = np.maximum(np.abs(y_true[mask]), eps)
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / denom)) * 100.0)

def fsi_vs_baseline(y_true, y_pred, y_base, eps=1e-12) -> float:
    rb = rmse(y_true, y_base)
    rm_ = rmse(y_true, y_pred)
    if rb < eps:
        return np.nan
    return float(1.0 - (rm_ / rb))

def metrics_row(name, y_true, y_pred, y_base=None):
    row = {
        "set": name,
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
        "sMAPE%": smape(y_true, y_pred),
        "MAPE_safe%": safe_mape(y_true, y_pred, thresh=10.0),
    }
    if y_base is not None:
        row["FSI_vs_DAM"] = fsi_vs_baseline(y_true, y_pred, y_base)
        row["RMSE_DAM_baseline"] = rmse(y_true, y_base)
    return row

# ---------------------------
# LOAD + BASIC PREP
# ---------------------------
df = pd.read_csv(cfg.DATA_PATH)
df[cfg.TS_COL] = pd.to_datetime(df[cfg.TS_COL])
df = df.sort_values(cfg.TS_COL).reset_index(drop=True)

# Identify a BM price column robustly
bm_price_candidates = ["BM_IMBALANCE_PRICE", "BM_PRICE", "BM_MCP", "IMBALANCE_PRICE", "BM_PRICE_MCP", "BM_Price"]
bm_price_col = next((c for c in bm_price_candidates if c in df.columns), None)

if bm_price_col is None:
    raise ValueError(
        "Δεν βρήκα BM price column. Βεβαιώσου ότι στο Final2026.csv υπάρχει μία από αυτές: "
        f"{bm_price_candidates}\n"
        "Αν έχεις άλλο όνομα (π.χ. 'BM_PRICE_X'), άλλαξε το bm_price_candidates."
    )

# Optional: drop BM price if you do NOT trust it as ex-ante
if (not cfg.INCLUDE_BM_IMBALANCE_PRICE) and (bm_price_col in df.columns):
    df = df.drop(columns=[bm_price_col])
    bm_price_col = None

# Time fields
df["day"] = df[cfg.TS_COL].dt.floor("D")
df["minute_of_day"] = df[cfg.TS_COL].dt.hour * 60 + df[cfg.TS_COL].dt.minute
df["mtu_idx"] = (df["minute_of_day"] // cfg.FREQ_MIN).astype(int)  # 0..95
df["block_id"] = (df[cfg.TS_COL].dt.hour // cfg.BLOCK_HOURS).astype(int)  # 0..5
df["dow"] = df[cfg.TS_COL].dt.dayofweek.astype(int)
df["hour"] = df[cfg.TS_COL].dt.hour.astype(int)

# Seasonality (ex-ante)
df["sin_hour"] = np.sin(2*np.pi*df["hour"]/24.0)
df["cos_hour"] = np.cos(2*np.pi*df["hour"]/24.0)
df["sin_dow"]  = np.sin(2*np.pi*df["dow"]/7.0)
df["cos_dow"]  = np.cos(2*np.pi*df["dow"]/7.0)

# ---------------------------
# D-1 SAME-MTU ALIGNMENT (Leakage-safe)
#   BM_price_Dm1(t) = BM_price(t - 1 day) matched to same timestamp-of-day
#   Implementation: make lookup where yesterday timestamp is shifted +1 day -> merges onto today timestamp.
# ---------------------------
if bm_price_col is not None:
    lookup_bm = df[[cfg.TS_COL, bm_price_col]].copy()
    lookup_bm[cfg.TS_COL] = lookup_bm[cfg.TS_COL] + pd.Timedelta(days=1)
    lookup_bm = lookup_bm.rename(columns={bm_price_col: f"{bm_price_col}_Dm1"})
    df = df.merge(lookup_bm, on=cfg.TS_COL, how="left")

# Optional: D-1 BM spread BMmDAMMCP if exists (often very strong as D-1 signal)
if "BMmDAMMCP" in df.columns:
    lookup_sp = df[[cfg.TS_COL, "BMmDAMMCP"]].copy()
    lookup_sp[cfg.TS_COL] = lookup_sp[cfg.TS_COL] + pd.Timedelta(days=1)
    lookup_sp = lookup_sp.rename(columns={"BMmDAMMCP": "BMmDAMMCP_Dm1"})
    df = df.merge(lookup_sp, on=cfg.TS_COL, how="left")
else:
    df["BMmDAMMCP_Dm1"] = np.nan  # keep column for consistent feature list (will be dropped if all NaN)

# Drop first day(s) where D-1 is missing
dm1_cols = []
if bm_price_col is not None:
    dm1_cols.append(f"{bm_price_col}_Dm1")
if "BMmDAMMCP_Dm1" in df.columns:
    dm1_cols.append("BMmDAMMCP_Dm1")

df = df.dropna(subset=[c for c in dm1_cols if c in df.columns]).reset_index(drop=True)

# Keep only full days with all 96 MTUs
counts = df.groupby("day")["mtu_idx"].nunique()
full_days = counts[counts == cfg.MTU_PER_DAY].index
df = df[df["day"].isin(full_days)].reset_index(drop=True)
df = df.sort_values(["day", "mtu_idx"]).reset_index(drop=True)

print("Total full days (after D-1 alignment):", df["day"].nunique())

# ---------------------------
# FEATURE SET (EX-ANTE ONLY)
# ---------------------------
required = [cfg.TARGET_COL, cfg.DAM_COL, "SystemLoad_DA_forecast", "Wind_DA_forecast", "Solar_DA_forecast"]
missing_req = [c for c in required if c not in df.columns]
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

FEATURES = [
    cfg.DAM_COL,
    "SystemLoad_DA_forecast",
    "Wind_DA_forecast",
    "Solar_DA_forecast",
    "sin_hour", "cos_hour",
    "sin_dow", "cos_dow",
    "mtu_idx",
    "block_id",
]

# Add D-1 aligned BM price + D-1 aligned BM spread if available
if bm_price_col is not None:
    FEATURES.append(f"{bm_price_col}_Dm1")
if "BMmDAMMCP_Dm1" in df.columns and (df["BMmDAMMCP_Dm1"].notna().any()):
    FEATURES.append("BMmDAMMCP_Dm1")

# Safety check
missF = [c for c in FEATURES if c not in df.columns]
if missF:
    raise ValueError(f"Feature columns missing from dataframe: {missF}")

# Target
if cfg.TARGET_MODE == "residual":
    df["y"] = df[cfg.TARGET_COL] - df[cfg.DAM_COL]
elif cfg.TARGET_MODE == "mcp":
    df["y"] = df[cfg.TARGET_COL]
else:
    raise ValueError("CFG.TARGET_MODE must be 'residual' or 'mcp'")

# DAM baseline for final MCP prediction
df["pred_DAM"] = df[cfg.DAM_COL].astype(float)

# ---------------------------
# SPLIT BY DAYS (CHRONOLOGICAL) — leakage-safe
# ---------------------------
days = sorted(df["day"].unique())
D = len(days)

n_train_days = int(D * cfg.TRAIN_FRAC)
n_val_days   = int(D * cfg.VAL_FRAC)
# ensure remainder goes to test
n_test_days  = D - n_train_days - n_val_days

train_days = days[:n_train_days]
val_days   = days[n_train_days:n_train_days + n_val_days]
test_days  = days[n_train_days + n_val_days:]

print("Split days:", len(train_days), len(val_days), len(test_days))

train_df = df[df["day"].isin(train_days)].copy()
val_df   = df[df["day"].isin(val_days)].copy()
test_df  = df[df["day"].isin(test_days)].copy()

X_tr = train_df[FEATURES].values.astype(np.float32)
y_tr = train_df["y"].values.astype(np.float32)

X_va = val_df[FEATURES].values.astype(np.float32)
y_va = val_df["y"].values.astype(np.float32)

X_te = test_df[FEATURES].values.astype(np.float32)
y_te = test_df["y"].values.astype(np.float32)

# ---------------------------
# TRAIN XGBOOST (EARLY STOPPING ON VAL) — no leakage
# ---------------------------
model = xgb.XGBRegressor(
    n_estimators=cfg.N_EST,
    learning_rate=cfg.LR,
    max_depth=cfg.MAX_DEPTH,
    subsample=cfg.SUBSAMPLE,
    colsample_bytree=cfg.COLSAMPLE,
    reg_lambda=cfg.REG_LAMBDA,
    random_state=cfg.SEED,
    tree_method="hist",
    # objective default is squared error for regressor; OK
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    verbose=False
)

# Predict residual or MCP
pred_y_va = model.predict(X_va).astype(np.float32)
pred_y_te = model.predict(X_te).astype(np.float32)

if cfg.TARGET_MODE == "residual":
    # MCP_hat = DAM + residual_hat
    pred_va = (val_df[cfg.DAM_COL].values.astype(np.float32) + pred_y_va).astype(np.float32)
    pred_te = (test_df[cfg.DAM_COL].values.astype(np.float32) + pred_y_te).astype(np.float32)
else:
    pred_va = pred_y_va
    pred_te = pred_y_te

# Baseline: DAM (already aligned on same timestamp)
base_va = val_df["pred_DAM"].values.astype(np.float32)
base_te = test_df["pred_DAM"].values.astype(np.float32)

true_va = val_df[cfg.TARGET_COL].values.astype(np.float32)
true_te = test_df[cfg.TARGET_COL].values.astype(np.float32)

# ---------------------------
# METRICS
# ---------------------------
metrics = []
metrics.append(metrics_row("VAL_DAM_BASELINE_MTU",  true_va, base_va, base_va))
metrics.append(metrics_row("VAL_XGB_MTU",           true_va, pred_va, base_va))
metrics.append(metrics_row("TEST_DAM_BASELINE_MTU", true_te, base_te, base_te))
metrics.append(metrics_row("TEST_XGB_MTU",          true_te, pred_te, base_te))

metrics_df = pd.DataFrame(metrics)
print("\n=== METRICS (MTU-level) ===")
print(metrics_df.sort_values("set").to_string(index=False))

# ---------------------------
# EXPORT PREDICTIONS CSV
# ---------------------------
val_out = val_df[[cfg.TS_COL, "day", "mtu_idx", "block_id", cfg.TARGET_COL, cfg.DAM_COL]].copy()
val_out["split"] = "VAL"
val_out["pred_DAM"] = base_va.astype(float)
val_out["pred_XGB"] = pred_va.astype(float)
val_out["err_DAM"] = (val_out["pred_DAM"].values - val_out[cfg.TARGET_COL].values).astype(float)
val_out["err_XGB"] = (val_out["pred_XGB"].values - val_out[cfg.TARGET_COL].values).astype(float)
val_out["abs_err_DAM"] = np.abs(val_out["err_DAM"].values).astype(float)
val_out["abs_err_XGB"] = np.abs(val_out["err_XGB"].values).astype(float)

test_out = test_df[[cfg.TS_COL, "day", "mtu_idx", "block_id", cfg.TARGET_COL, cfg.DAM_COL]].copy()
test_out["split"] = "TEST"
test_out["pred_DAM"] = base_te.astype(float)
test_out["pred_XGB"] = pred_te.astype(float)
test_out["err_DAM"] = (test_out["pred_DAM"].values - test_out[cfg.TARGET_COL].values).astype(float)
test_out["err_XGB"] = (test_out["pred_XGB"].values - test_out[cfg.TARGET_COL].values).astype(float)
test_out["abs_err_DAM"] = np.abs(test_out["err_DAM"].values).astype(float)
test_out["abs_err_XGB"] = np.abs(test_out["err_XGB"].values).astype(float)

preds_df = pd.concat([val_out, test_out], axis=0).reset_index(drop=True)
preds_df.to_csv(cfg.OUT_PRED_CSV, index=False)
metrics_df.to_csv(cfg.OUT_METRICS_CSV, index=False)

print("\nSaved:")
print(" - Predictions:", cfg.OUT_PRED_CSV)
print(" - Metrics    :", cfg.OUT_METRICS_CSV)

# ---------------------------
# SANITY CHECK (1 test day, print every 2 hours)
# ---------------------------
if len(test_days) > 0:
    d0 = test_days[0]
    g = test_out[test_out["day"] == d0].sort_values("mtu_idx")
    print("\n" + "-" * 120)
    print("SANITY CHECK — First TEST day:", pd.to_datetime(d0).date().isoformat())
    print("mtu_idx | True | DAM | XGB")
    print("-" * 120)
    for t in range(0, 96, 8):
        row = g[g["mtu_idx"] == t].iloc[0]
        print(f"{t:>6} | {row[cfg.TARGET_COL]:>6.2f} | {row[cfg.DAM_COL]:>6.2f} | {row['pred_XGB']:>6.2f}")
    print("-" * 120)

preds_df.head(12)


Total full days (after D-1 alignment): 102
Split days: 71 15 16

=== METRICS (MTU-level) ===
                  set      MAE     RMSE       R2   sMAPE%  MAPE_safe%  FSI_vs_DAM  RMSE_DAM_baseline
TEST_DAM_BASELINE_MTU 4.525553 5.858458 0.982900 8.315739    5.424564    0.000000           5.858458
         TEST_XGB_MTU 4.251844 5.876886 0.982793 8.646096    4.937671   -0.003145           5.858458
 VAL_DAM_BASELINE_MTU 3.639652 5.825755 0.969437 3.600003    3.686664    0.000000           5.825755
          VAL_XGB_MTU 4.171524 6.206499 0.965312 4.373544    4.151493   -0.065355           5.825755

Saved:
 - Predictions: /content/xgb_mtu_preds.csv
 - Metrics    : /content/xgb_mtu_metrics.csv

------------------------------------------------------------------------------------------------------------------------
SANITY CHECK — First TEST day: 2026-01-04
mtu_idx | True | DAM | XGB
-------------------------------------------------------------------------------------------------------------------

,DELIVERY_MTU,day,mtu_idx,block_id,MCP,DAM_MCP,split,pred_DAM,pred_XGB,err_DAM,err_XGB,abs_err_DAM,abs_err_XGB
0,2025-12-18 00:00:00,2025-12-18,0,0,119.60,121.03,VAL,121.029999,115.453270,1.429999,-4.146730,1.429999,4.146730
1,2025-12-18 00:15:00,2025-12-18,1,0,115.17,116.37,VAL,116.370003,114.003387,1.200003,-1.166613,1.200003,1.166613
2,2025-12-18 00:30:00,2025-12-18,2,0,98.62,102.26,VAL,102.260002,100.428879,3.640002,1.808879,3.640002,1.808879
3,2025-12-18 00:45:00,2025-12-18,3,0,93.74,96.74,VAL,96.739998,96.816017,2.999998,3.076017,2.999998,3.076017
4,2025-12-18 01:00:00,2025-12-18,4,0,119.39,119.60,VAL,119.599998,117.689423,0.209998,-1.700577,0.209998,1.700577
5,2025-12-18 01:15:00,2025-12-18,5,0,102.00,103.09,VAL,103.089996,102.651649,1.089996,0.651649,1.089996,0.651649
6,2025-12-18 01:30:00,2025-12-18,6,0,100.92,102.05,VAL,102.050003,101.121071,1.130003,0.201071,1.130003,0.201071
7,2025-12-18 01:45:00,2025-12-18,7,0,103.46,101.50,VAL,101.500000,101.367691,-1.960000,-2.092309,1.960000,2.092309
8,2025-12-18 02:00:00,2025-12-18,8,0,96.78,96.78,VAL,96.779999,96.353043,-0.000001,-0.426957,0.000001,0.426957
9,2025-12-18 02:15:00,2025-12-18,9,0,105.52,99.76,VAL,99.760002,100.336456,-5.759998,-5.183544,5.759998,5.183544


Συμπέρασμα για το XGBoost σε MTU επίπεδο

Παρότι το μοντέλο XGBoost αξιοποιεί ex-ante προβλέψεις (ζήτηση, ΑΠΕ) καθώς και πληροφορία από την αγορά εξισορρόπησης της προηγούμενης ημέρας (D−1 BM), δεν καταφέρνει να βελτιώσει ουσιαστικά το DAM baseline σε ανάλυση 15 λεπτών. Αυτό δείχνει ότι, όταν το μοντέλο δεν διαθέτει χρονική μνήμη, η διαθέσιμη πληροφορία δεν είναι επαρκής για να αποτυπώσει τις ενδοημερήσιες διακυμάνσεις της τιμής.

Σύγκριση με το LSTM

Αντίθετα, το residual LSTM παρουσιάζει ξεκάθαρη βελτίωση σε σχέση με το DAM, γεγονός που επιβεβαιώνει ότι οι χρονικές εξαρτήσεις και τα intraday patterns παίζουν καθοριστικό ρόλο στην πρόβλεψη τιμών σε ανάλυση 15 λεπτών. Η δυνατότητα του LSTM να αξιοποιεί ιστορική πληροφορία το καθιστά πιο κατάλληλο για τη σύλληψη της δυναμικής της αγοράς σε λεπτό χρονικό ορίζοντα.